In [2]:
import numpy as np

lst = [1, 2, 3, 4, 5]
np_arr = np.array(lst)

print(np.__version__)
print(np_arr)
print(len(np_arr))
print(np_arr * 2)
print(np_arr + 2)

2.5.5
[1 2 3 4 5]
5
[ 2  4  6  8 10]
[3 4 5 6 7]


In [8]:
import re
from collections import Counter
from urllib.request import Request, urlopen

# The Gutenberg endpoint is returning HTTP 503; use its public GitHub mirror.
romeo_and_juliet_url = "https://raw.githubusercontent.com/GITenberg/Romeo-and-Juliet_1513/master/1513-0.txt"
request = Request(romeo_and_juliet_url, headers={"User-Agent": "Python learning exercise"})

with urlopen(request, timeout=30) as response:
    romeo_and_juliet = response.read().decode("utf-8")

words = re.findall(r"\b[\w']+\b", romeo_and_juliet.lower())
word_frequency = Counter(words)

print("10 most frequent words in Romeo and Juliet:")
for word, count in word_frequency.most_common(10):
    print(f"{word}: {count}")

10 most frequent words in Romeo and Juliet:
the: 876
and: 811
i: 658
to: 626
a: 548
of: 519
in: 396
is: 373
that: 369
you: 368


In [7]:
import json
import os
import re
from collections import Counter
from statistics import mean, median, pstdev
from urllib.error import HTTPError
from urllib.request import Request, urlopen

cats_api = "https://api.thecatapi.com/v1/breeds"
headers = {"User-Agent": "Python learning exercise"}
cat_api_key = os.getenv("CAT_API_KEY")
if cat_api_key:
    headers["x-api-key"] = cat_api_key

request = Request(cats_api, headers=headers)
try:
    with urlopen(request, timeout=30) as response:
        cat_breeds = json.loads(response.read().decode("utf-8"))
except HTTPError as error:
    if error.code == 403 and not cat_api_key:
        raise RuntimeError(
            "TheCatAPI denied this request. Get a free API key at https://thecatapi.com/ "
            "and set it in the CAT_API_KEY environment variable."
        ) from error
    raise

def range_midpoint(value):
    numbers = [float(number) for number in re.findall(r"\d+(?:\.\d+)?", str(value))]
    return sum(numbers) / len(numbers) if numbers else None

def print_statistics(label, values):
    print(f"{label} (one midpoint per breed):")
    print(f"  min: {min(values):.2f}")
    print(f"  max: {max(values):.2f}")
    print(f"  mean: {mean(values):.2f}")
    print(f"  median: {median(values):.2f}")
    print(f"  standard deviation: {pstdev(values):.2f}")

weights_kg = [
    midpoint
    for breed in cat_breeds
    if (midpoint := range_midpoint(breed.get("weight", {}).get("metric"))) is not None
]
lifespans_years = [
    midpoint
    for breed in cat_breeds
    if (midpoint := range_midpoint(breed.get("life_span"))) is not None
]

print_statistics("Weight in kilograms", weights_kg)
print_statistics("Lifespan in years", lifespans_years)

# Each row counts breeds by the API's country of origin and breed name.
country_breed_frequency = Counter(
    (breed.get("origin") or "Unknown", breed.get("name") or "Unknown")
    for breed in cat_breeds
)
country_frequency = Counter(country for country, _ in country_breed_frequency)

print("\nBreed counts by country of origin:")
for country, count in sorted(country_frequency.items()):
    print(f"{country}: {count}")

print("\nCountry, breed, frequency:")
for (country, breed), count in sorted(country_breed_frequency.items()):
    print(f"{country}, {breed}, {count}")

RuntimeError: TheCatAPI denied this request. Get a free API key at https://thecatapi.com/ and set it in the CAT_API_KEY environment variable.

In [9]:
import json
from collections import Counter
from urllib.request import Request, urlopen

# The exercise's restcountries.eu endpoint is discontinued. Use a public
# country list for languages and World Bank data for the latest populations.
countries_url = "https://raw.githubusercontent.com/mledoze/countries/master/countries.json"
population_url = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&per_page=400&mrv=1"
headers = {"User-Agent": "Python learning exercise"}

with urlopen(Request(countries_url, headers=headers), timeout=30) as response:
    countries = json.loads(response.read().decode("utf-8"))
with urlopen(Request(population_url, headers=headers), timeout=30) as response:
    population_response = json.loads(response.read().decode("utf-8"))

population_by_code = {
    record["countryiso3code"]: record["value"]
    for record in population_response[1]
    if record.get("countryiso3code") and isinstance(record.get("value"), (int, float))
}
country_populations = [
    (country["name"]["common"], population_by_code[country["cca3"]])
    for country in countries
    if country.get("cca3") in population_by_code
]
largest_countries = sorted(country_populations, key=lambda item: item[1], reverse=True)[:10]

print("10 largest countries by latest available World Bank population:")
for country_name, population in largest_countries:
    print(f"{country_name}: {population:,}")

# This dataset does not report speaker counts, so language frequency means
# the number of countries listing each language.
language_frequency = Counter(
    language
    for country in countries
    for language in (country.get("languages") or {}).values()
)
print("\n10 most widely listed languages (number of countries):")
for language, count in language_frequency.most_common(10):
    print(f"{language}: {count} countries")

print(f"\nTotal distinct languages: {len(language_frequency)}")

10 largest countries by latest available World Bank population:
India: 1,463,865,525
China: 1,406,585,000
United States: 341,784,857
Indonesia: 285,721,236
Pakistan: 255,219,554
Nigeria: 237,527,782
Brazil: 212,812,405
Bangladesh: 175,686,899
Russia: 143,513,328
Ethiopia: 135,472,051

10 most widely listed languages (number of countries):
English: 91 countries
French: 46 countries
Arabic: 25 countries
Spanish: 24 countries
Portuguese: 10 countries
Russian: 8 countries
Dutch: 7 countries
German: 5 countries
Chinese: 5 countries
Tswana: 4 countries

Total distinct languages: 155


In [10]:
from urllib.parse import urljoin
from urllib.request import Request, urlopen

from bs4 import BeautifulSoup

uci_url = "https://archive.ics.uci.edu/datasets"
request = Request(uci_url, headers={"User-Agent": "Mozilla/5.0 (Python learning exercise)"})
with urlopen(request, timeout=30) as response:
    uci_html = response.read().decode("utf-8")

soup = BeautifulSoup(uci_html, "html.parser")
dataset_links = soup.select('a[href^="/dataset/"]')
datasets = []
seen_urls = set()

for link in dataset_links:
    dataset_name = link.get_text(" ", strip=True)
    dataset_url = urljoin(uci_url, link.get("href", ""))
    if dataset_name and dataset_url not in seen_urls:
        seen_urls.add(dataset_url)
        datasets.append((dataset_name, dataset_url))

print(f"Found {len(datasets)} dataset links on the current UCI results page:")
for dataset_name, dataset_url in datasets:
    print(f"{dataset_name}: {dataset_url}")

Found 10 dataset links on the current UCI results page:
Iris: https://archive.ics.uci.edu/dataset/53/iris
Heart Disease: https://archive.ics.uci.edu/dataset/45/heart+disease
Wine Quality: https://archive.ics.uci.edu/dataset/186/wine+quality
Bank Marketing: https://archive.ics.uci.edu/dataset/222/bank+marketing
Student Performance: https://archive.ics.uci.edu/dataset/320/student+performance
Breast Cancer Wisconsin (Diagnostic): https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic
Adult: https://archive.ics.uci.edu/dataset/2/adult
Online Retail: https://archive.ics.uci.edu/dataset/352/online+retail
Wine: https://archive.ics.uci.edu/dataset/109/wine
Car Evaluation: https://archive.ics.uci.edu/dataset/19/car+evaluation
